# Make YOUR cloned voice (free, no account needed)\n\n**Read these 6 sentences at a natural pace — don't rush, each takes ~12-18s.** These are the SAME 6 the notebook clones in your voice, so real and cloned clips match exactly. Record 3 English first, then 3 Hindi, one line = one clip:\n\n1. My account number is three two seven four one nine, and I am calling to verify the mailing address currently on file, because I recently moved to a new apartment and I want to make sure nothing gets sent to the old one.\n2. Could you please confirm the exact delivery time for my package tomorrow afternoon? I have a meeting until three, so I would prefer if the courier could come sometime after four, if that is possible to arrange.\n3. I would like to formally report an issue with my recent bank transaction — the amount deducted does not match what I authorized, and I need this resolved before the end of the week.\n4. मेरा खाता नंबर तीन दो सात चार एक नौ है, और मैं यह सुनिश्चित करना चाहता हूँ कि फ़ाइल पर मौजूद पता सही है, क्योंकि मैं हाल ही में एक नए घर में शिफ्ट हुआ हूँ।\n5. क्या आप कृपया बता सकते हैं कि कल दोपहर मेरे पार्सल की डिलीवरी ठीक किस समय होगी? मुझे शाम चार बजे तक एक मीटिंग है, इसलिए उसके बाद आना ज़्यादा सही रहेगा।\n6. मैं अपने हाल के बैंक लेनदेन को लेकर एक गंभीर समस्या की रिपोर्ट करना चाहता हूँ — जो राशि काटी गई है वह मेरी अनुमति से मेल नहीं खाती, और मुझे यह जल्द से जल्द ठीक करवाना है।\n\n**Setup:** Runtime → Change runtime type → **T4 GPU** (don't skip — CPU is very slow).\nThen run cells 0→1→2→3.

In [ ]:
# @title 0. Install XTTS-v2 (open-source voice cloning, no account, no paywall)
# IMPORTANT: use a GPU runtime first! Runtime -> Change runtime type -> T4 GPU.
# Free zero-shot voice cloning on Colab. No signup, no card, no daily limit lottery.
!pip install -q TTS
# If the install above errors, uncomment and run this (actively maintained fork, same import):
# !pip install -q coqui-tts
print("XTTS installed")

In [ ]:
# @title 1. Set your name + upload your reference voice (real recording)
# IMPORTANT: your real recordings MUST have said the exact sentences in cell 2 below.
# Check GPU before continuing: if the next line prints "GPU: False", first do
# Runtime -> Change runtime type -> T4 GPU, then re-run this cell.
import torch
print("GPU:", torch.cuda.is_available())
import os, pathlib, glob, shutil

YOUR_NAME = "team_member_name"   # <-- CHANGE this to your name

os.makedirs(f"/content/clones/{YOUR_NAME}", exist_ok=True)

from google.colab import files
print("\nUpload ONE clear 10-20s real clip of you speaking (the ref voice for cloning).")
used = files.upload()
if len(used) != 1:
    # warn loudly instead of silently picking the wrong file
    raise SystemExit(f"Expected EXACTLY one uploaded file, got {len(used)}: {list(used)}. Cancel and retry one file.")
ref = list(used.keys())[0]
print("Reference set:", ref)
REF_PATH = ref

In [ ]:
# @title 2. Generate your cloned clips (same sentences, in your voice)
# Each clone matches one sentence -> real + cloned pairs with the SAME words.
# NOTE: if you did NOT read these exact sentences in your real recording, later
# real-vs-cloned pairing will be broken. Re-record real clips to match them.
import os
os.environ["COQUI_TOS_AGREED"] = "1"   # bypass first-run license prompt (would hang in Colab)
import torch, glob
from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda" if torch.cuda.is_available() else "cpu")

sentences = [
    # English (3)
    "My account number is three two seven four one nine, and I am calling to verify the mailing address currently on file, because I recently moved to a new apartment and I want to make sure nothing gets sent to the old one.",
    "Could you please confirm the exact delivery time for my package tomorrow afternoon? I have a meeting until three, so I would prefer if the courier could come sometime after four, if that is possible to arrange.",
    "I would like to formally report an issue with my recent bank transaction — the amount deducted does not match what I authorized, and I need this resolved before the end of the week.",
    # Hindi (3)
    "मेरा खाता नंबर तीन दो सात चार एक नौ है, और मैं यह सुनिश्चित करना चाहता हूँ कि फ़ाइल पर मौजूद पता सही है, क्योंकि मैं हाल ही में एक नए घर में शिफ्ट हुआ हूँ।",
    "क्या आप कृपया बता सकते हैं कि कल दोपहर मेरे पार्सल की डिलीवरी ठीक किस समय होगी? मुझे शाम चार बजे तक एक मीटिंग है, इसलिए उसके बाद आना ज़्यादा सही रहेगा।",
    "मैं अपने हाल के बैंक लेनदेन को लेकर एक गंभीर समस्या की रिपोर्ट करना चाहता हूँ — जो राशि काटी गई है वह मेरी अनुमति से मेल नहीं खाती, और मुझे यह जल्द से जल्द ठीक करवाना है।",
]

for idx, text in enumerate(sentences, start=1):
    out = f"/content/clones/{YOUR_NAME}/clone_{idx:02d}.wav"
    tts.tts_to_file(
        text=text,
        speaker_wav=REF_PATH,
        language="en" if idx <= 3 else "hi",
        file_path=out,
    )
    print("wrote", out)

print("\nDone. Files:")
for f in sorted(glob.glob(f"/content/clones/{YOUR_NAME}/*.wav")):
    print("  ", f)

In [ ]:
# @title 3. Download your cloned clips + your real clips (deliver 12 files)
from google.colab import files
import glob, os
for f in sorted(glob.glob(f"/content/clones/{YOUR_NAME}/*.wav")):
    files.download(f)

print("\nNow put your 6 REAL clips (from your phone, the exact 6 sentences above)")
print("and these 6 CLONED clips into one folder named after you:")
print("  <yourname>/  real_01..06.wav  +  clone_01..06.wav   (= 12 files)")